# Evaluate & Compare (Sections 14-29)

Loads whichever model checkpoints are currently saved on Drive by the `0X_train_*.ipynb` notebooks and runs the full evaluation pipeline: test metrics, confusion matrices, ROC/PR curves, computational comparison, prediction analysis, Grad-CAM, error analysis, confidence analysis, robustness testing, McNemar's test, and the auto-generated findings/limitations/conclusion.


## Environment + Drive

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None, pip_name=None):
    import_name = import_name or pkg
    pip_name = pip_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

for pkg, imp in [("timm", "timm"), ("opencv-python", "cv2"), ("scikit-learn", "sklearn")]:
    ensure(pkg, imp, pkg)

import torch, torchvision
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name        :", torch.cuda.get_device_name(0))
    print("CUDA version    :", torch.version.cuda)
else:
    print("No GPU detected - Runtime > Change runtime type > GPU (T4) is recommended.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device     :", DEVICE)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Configuration

In [ ]:
import os

DATASET_DIR = "/content/dataset"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/deepfake_results"
OUTPUT_DIR = "/content/deepfake_results"   # local copies of figures/tables for this session

IMG_SIZE, BATCH_SIZE = 224, 32
RANDOM_SEED, THRESHOLD, NUM_WORKERS = 42, 0.5, 0

MODELS_TO_RUN = ["resnet18", "resnet50", "efficientnet_b0", "xception"]

for sub in ["plots", "confusion_matrices", "roc_curves", "pr_curves",
            "gradcam", "predictions", "tables", "robustness"]:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
print("Seed set to", RANDOM_SEED)

## Dataset (test split needed for evaluation)

In [ ]:

RUN_OPTION_C = False   # set True to use this option

if RUN_OPTION_C:
    from google.colab import files
    print("Select your kaggle.json token")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    import shutil
    shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download",
                     "-d", "manjilkarki/deepfake-and-real-images", "-p", "/content"], check=True)
    import zipfile
    with zipfile.ZipFile("/content/deepfake-and-real-images.zip", "r") as z:
        z.extractall(DATASET_DIR)
    print("Downloaded and extracted to", DATASET_DIR)

In [ ]:
def find_split_root(base_dir):
    target_names = {"train", "validation", "valid", "test"}
    for root, dirs, _ in os.walk(base_dir):
        lower_dirs = {d.lower() for d in dirs}
        if len(lower_dirs & target_names) >= 2:
            return root
    return base_dir

assert os.path.exists(DATASET_DIR), f"DATASET_DIR does not exist: {DATASET_DIR}. Run an Option A/B/C cell first."
SPLIT_ROOT = find_split_root(DATASET_DIR)

def resolve_split_dir(root, names):
    for n in names:
        p = os.path.join(root, n)
        if os.path.isdir(p):
            return p
    return None

TRAIN_DIR = resolve_split_dir(SPLIT_ROOT, ["Train", "train"])
VAL_DIR   = resolve_split_dir(SPLIT_ROOT, ["Validation", "validation", "Valid", "valid", "Val", "val"])
TEST_DIR  = resolve_split_dir(SPLIT_ROOT, ["Test", "test"])
for label, d in [("TRAIN_DIR", TRAIN_DIR), ("VAL_DIR", VAL_DIR), ("TEST_DIR", TEST_DIR)]:
    print(label, "->", d)
    if d is None:
        raise RuntimeError(f"Could not locate {label} under {SPLIT_ROOT}.")

def resolve_class_dir(split_dir, names):
    for n in names:
        p = os.path.join(split_dir, n)
        if os.path.isdir(p):
            return p
    return None

def class_dirs(split_dir):
    return (resolve_class_dir(split_dir, ["Real", "real", "REAL"]),
            resolve_class_dir(split_dir, ["Fake", "fake", "FAKE"]))

TRAIN_REAL_DIR, TRAIN_FAKE_DIR = class_dirs(TRAIN_DIR)
VAL_REAL_DIR, VAL_FAKE_DIR = class_dirs(VAL_DIR)
TEST_REAL_DIR, TEST_FAKE_DIR = class_dirs(TEST_DIR)
print("Resolved class folders OK.")

## Test-set loader

In [ ]:
import glob, random
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class DeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform):
        exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.JPG", "*.JPEG", "*.PNG")
        real_files, fake_files = [], []
        for e in exts:
            real_files += glob.glob(os.path.join(real_dir, e)) if real_dir else []
            fake_files += glob.glob(os.path.join(fake_dir, e)) if fake_dir else []
        self.samples = [(f, 0) for f in real_files] + [(f, 1) for f in fake_files]
        random.Random(RANDOM_SEED).shuffle(self.samples)
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self.transform(Image.open(path).convert("RGB"))
        import torch
        return img, torch.tensor(label, dtype=torch.float32), path

test_ds = DeepfakeDataset(TEST_REAL_DIR, TEST_FAKE_DIR, eval_transform)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("Test samples:", len(test_ds))

## Model architecture (must match training notebooks)

In [ ]:
import torch.nn as nn
import torchvision.models as tvm
try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False

class BinaryClassifier(nn.Module):
    def __init__(self, backbone, target_layer):
        super().__init__()
        self.backbone = backbone
        self._target_layer = target_layer
    def forward(self, x):
        return self.backbone(x).squeeze(-1)
    def get_target_layer(self):
        return self._target_layer

def build_resnet18():
    m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return BinaryClassifier(m, m.layer4[-1])

def build_resnet50():
    m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return BinaryClassifier(m, m.layer4[-1])

def build_efficientnet_b0():
    m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    return BinaryClassifier(m, m.features[-1])

def build_xception():
    if not HAS_TIMM:
        raise RuntimeError("timm is required for Xception.")
    m = timm.create_model("xception", pretrained=True, num_classes=1)
    target_layer = None
    for _, module in m.named_modules():
        if isinstance(module, nn.Conv2d):
            target_layer = module
    return BinaryClassifier(m, target_layer)

MODEL_BUILDERS = {
    "resnet18": build_resnet18, "resnet50": build_resnet50,
    "efficientnet_b0": build_efficientnet_b0, "xception": build_xception,
}

## Load available checkpoints from Drive

In [ ]:
import torch

available_models = {}
for name in MODELS_TO_RUN:
    ckpt = os.path.join(DRIVE_RESULTS_DIR, "models", f"{name}_best.pth")
    if os.path.exists(ckpt):
        model = MODEL_BUILDERS[name]().to(DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        model.eval()
        available_models[name] = model
        print(f"Loaded {name} from {ckpt}")
    else:
        print(f"No checkpoint yet for {name} at {ckpt} - it will be skipped below.")

MODELS_TO_RUN = list(available_models.keys())
trained_models = available_models
assert len(trained_models) > 0, "No trained checkpoints found on Drive yet - train at least one model first."
print("\nEvaluating models:", MODELS_TO_RUN)

## Training curves (from saved history)

In [ ]:
import json
import matplotlib.pyplot as plt

histories = {}
for name in MODELS_TO_RUN:
    hist_path = os.path.join(DRIVE_RESULTS_DIR, "history", f"{name}_history.json")
    if os.path.exists(hist_path):
        with open(hist_path) as f:
            histories[name] = json.load(f)

if histories:
    fig, axes = plt.subplots(len(histories), 2, figsize=(12, 4 * len(histories)))
    if len(histories) == 1:
        axes = axes.reshape(1, -1)
    for i, name in enumerate(histories):
        h = histories[name]
        ep = range(1, len(h["train_loss"]) + 1)
        axes[i, 0].plot(ep, h["train_loss"], label="Train")
        axes[i, 0].plot(ep, h["val_loss"], label="Validation")
        axes[i, 0].set_title(f"{name} - Loss"); axes[i, 0].legend()
        axes[i, 1].plot(ep, h["train_acc"], label="Train")
        axes[i, 1].plot(ep, h["val_acc"], label="Validation")
        axes[i, 1].set_title(f"{name} - Accuracy"); axes[i, 1].legend()
    fig.suptitle("Figure 3: Training / Validation curves per model")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plots", "fig3_training_curves.png"), dpi=300)
    plt.show()
else:
    print("No history files found on Drive.")

## 14. Test Evaluation

In [ ]:
import time
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve,
                              average_precision_score)

@torch.no_grad()
def evaluate_on_loader(model, loader):
    model.eval()
    all_probs, all_labels, all_paths = [], [], []
    for imgs, labels, paths in loader:
        imgs = imgs.to(DEVICE)
        probs = torch.sigmoid(model(imgs)).cpu().numpy()
        all_probs.extend(probs.tolist()); all_labels.extend(labels.numpy().tolist()); all_paths.extend(paths)
    return np.array(all_probs), np.array(all_labels), all_paths

def compute_classification_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(set(y_true.tolist())) > 1 else float("nan"),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else float("nan"),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else float("nan"),
        "fnr": fn / (fn + tp) if (fn + tp) > 0 else float("nan"),
        "confusion_matrix": (tn, fp, fn, tp), "y_pred": y_pred,
    }

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def model_size_mb(model):
    tmp_path = "/tmp/_tmp_model.pth"
    torch.save(model.state_dict(), tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 ** 2)
    os.remove(tmp_path)
    return size_mb

@torch.no_grad()
def measure_inference_time(model, loader, n_batches=10):
    model.eval(); times = []
    it = iter(loader)
    for _ in range(min(n_batches, len(loader))):
        imgs, _, _ = next(it)
        imgs = imgs.to(DEVICE)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        _ = model(imgs)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        times.append((time.time() - t0) / imgs.size(0))
    return float(np.mean(times)) if times else float("nan")

test_results = {}
for name, model in trained_models.items():
    probs, labels, paths = evaluate_on_loader(model, test_loader)
    metrics = compute_classification_metrics(labels, probs)
    total_p, trainable_p = count_params(model)
    size_mb = model_size_mb(model)
    inf_time = measure_inference_time(model, test_loader)
    test_results[name] = {
        **metrics, "y_true": labels, "y_prob": probs, "paths": paths,
        "params": total_p, "trainable_params": trainable_p, "model_size_mb": size_mb,
        "inference_time_s": inf_time, "throughput_img_s": 1.0 / inf_time if inf_time > 0 else float("nan"),
    }
    print(f"{name:16s} acc={metrics['accuracy']:.4f} f1={metrics['f1']:.4f} auc={metrics['roc_auc']:.4f} "
          f"params={total_p/1e6:.2f}M size={size_mb:.1f}MB inf_time={inf_time*1000:.2f}ms/img")

## 15. Confusion Matrices (Figure 7)

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, len(MODELS_TO_RUN), figsize=(5 * len(MODELS_TO_RUN), 4.5))
if len(MODELS_TO_RUN) == 1: axes = [axes]
for ax, name in zip(axes, MODELS_TO_RUN):
    tn, fp, fn, tp = test_results[name]["confusion_matrix"]
    cm = np.array([[tn, fp], [fn, tp]])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["REAL", "FAKE"], yticklabels=["REAL", "FAKE"], cbar=False)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
fig.suptitle("Figure 7: Confusion matrices"); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "fig7_confusion_matrices.png"), dpi=300)
plt.show()

## 16. ROC Analysis (Figure 5)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name in MODELS_TO_RUN:
    y_true, y_prob = test_results[name]["y_true"], test_results[name]["y_prob"]
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={test_results[name]['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Figure 5: ROC Curves"); ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "fig5_roc_curves.png"), dpi=300)
plt.show()

## 17. Precision-Recall Analysis (Figure 6)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name in MODELS_TO_RUN:
    y_true, y_prob = test_results[name]["y_true"], test_results[name]["y_prob"]
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax.plot(rec, prec, label=f"{name} (AP={ap:.3f})")
    test_results[name]["average_precision"] = ap
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Figure 6: Precision-Recall Curves"); ax.legend(loc="lower left")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "fig6_pr_curves.png"), dpi=300)
plt.show()

### Final Performance / Computational Tables

In [ ]:
import pandas as pd

perf_rows, comp_rows = [], []
for name in MODELS_TO_RUN:
    r = test_results[name]
    perf_rows.append({"Model": name, "Accuracy": r["accuracy"], "Precision": r["precision"],
                       "Recall": r["recall"], "F1": r["f1"], "ROC-AUC": r["roc_auc"],
                       "Specificity": r["specificity"], "Sensitivity": r["sensitivity"],
                       "FPR": r["fpr"], "FNR": r["fnr"]})
    comp_rows.append({"Model": name, "Parameters": r["params"], "Trainable Parameters": r["trainable_params"],
                       "Model Size (MB)": round(r["model_size_mb"], 2),
                       "Inference Time (ms/img)": round(r["inference_time_s"] * 1000, 3),
                       "Throughput (img/s)": round(r["throughput_img_s"], 2)})

perf_df = pd.DataFrame(perf_rows)
comp_df = pd.DataFrame(comp_rows)
perf_df.to_csv(os.path.join(OUTPUT_DIR, "tables", "model_performance.csv"), index=False)
comp_df.to_csv(os.path.join(OUTPUT_DIR, "tables", "computational_comparison.csv"), index=False)
display(perf_df); display(comp_df)

## 18. Computational Comparison (Figure 8) + Figure 4

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(comp_df["Model"], comp_df["Parameters"] / 1e6, color="#55A868")
axes[0].set_ylabel("Parameters (millions)"); axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(comp_df["Model"], comp_df["Inference Time (ms/img)"], color="#C44E52")
axes[1].set_ylabel("Inference time (ms/image)"); axes[1].tick_params(axis="x", rotation=30)
fig.suptitle("Figure 8: Computational comparison"); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "fig8_computational_comparison.png"), dpi=300)
plt.show()

metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metrics_to_plot)); width = 0.8 / len(MODELS_TO_RUN)
for i, name in enumerate(MODELS_TO_RUN):
    row = perf_df[perf_df["Model"] == name].iloc[0]
    ax.bar(x + i * width, [row[m] for m in metrics_to_plot], width, label=name)
ax.set_xticks(x + width * (len(MODELS_TO_RUN) - 1) / 2); ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.05); ax.set_title("Figure 4: Model performance comparison"); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "fig4_performance_comparison.png"), dpi=300)
plt.show()

## 19. Prediction Probability Analysis

In [ ]:
ANALYSIS_MODEL = MODELS_TO_RUN[0]
r = test_results[ANALYSIS_MODEL]
y_true, y_prob, paths, y_pred = r["y_true"], r["y_prob"], r["paths"], r["y_pred"]
correct_mask = (y_true == y_pred)
confidence = np.where(y_pred == 1, y_prob, 1 - y_prob)

def show_examples(indices, title):
    n = min(len(indices), 6)
    if n == 0:
        print(title, "- no examples found."); return
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
    if n == 1: axes = [axes]
    for ax, idx in zip(axes, indices[:n]):
        img = Image.open(paths[idx]).convert("RGB")
        ax.imshow(img); ax.axis("off")
        gt = "FAKE" if y_true[idx] == 1 else "REAL"
        pred = "FAKE" if y_pred[idx] == 1 else "REAL"
        ax.set_title(f"GT:{gt} Pred:{pred}\nFAKE p={y_prob[idx]:.3f}", fontsize=8)
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "predictions", title.replace(" ", "_") + ".png"), dpi=300)
    plt.show()

correct_idx = np.where(correct_mask)[0]; incorrect_idx = np.where(~correct_mask)[0]
show_examples(correct_idx[np.argsort(-confidence[correct_idx])].tolist(), f"{ANALYSIS_MODEL} - Highest confidence CORRECT")
show_examples(correct_idx[np.argsort(confidence[correct_idx])].tolist(), f"{ANALYSIS_MODEL} - Lowest confidence CORRECT")
show_examples(incorrect_idx[np.argsort(-confidence[incorrect_idx])].tolist(), f"{ANALYSIS_MODEL} - Highest confidence INCORRECT")

## 20-21. Grad-CAM Implementation + Comparison (Figure 9)

In [ ]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.target_layer = target_layer
        self.activations = None; self.gradients = None
        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)
    def _save_activation(self, module, inp, out): self.activations = out.detach()
    def _save_gradient(self, module, grad_in, grad_out): self.gradients = grad_out[0].detach()
    def __call__(self, input_tensor):
        self.model.zero_grad()
        logit = self.model(input_tensor)
        logit.backward()
        weights = self.gradients[0].mean(dim=(1, 2))
        cam = torch.relu((weights[:, None, None] * self.activations[0]).sum(dim=0)).cpu().numpy()
        if cam.max() > 0: cam = cam / cam.max()
        return cam, torch.sigmoid(logit).item()
    def remove(self): self.fwd_handle.remove(); self.bwd_handle.remove()

def overlay_cam_on_image(pil_img, cam, img_size=IMG_SIZE, alpha=0.45):
    img = np.array(pil_img.resize((img_size, img_size))).astype(np.float32) / 255.0
    cam_resized = cv2.resize(cam, (img_size, img_size))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.clip((1 - alpha) * img + alpha * heatmap, 0, 1)

def gradcam_for_image(model, img_path, img_size=IMG_SIZE):
    pil_img = Image.open(img_path).convert("RGB")
    tensor = eval_transform(pil_img).unsqueeze(0).to(DEVICE)
    cam_engine = GradCAM(model, model.get_target_layer())
    cam, fake_prob = cam_engine(tensor)
    cam_engine.remove()
    return overlay_cam_on_image(pil_img, cam, img_size), fake_prob

def find_example(category, y_true, y_pred, paths):
    if category == "correct_real": idx = np.where((y_true == 0) & (y_pred == 0))[0]
    elif category == "correct_fake": idx = np.where((y_true == 1) & (y_pred == 1))[0]
    elif category == "false_positive": idx = np.where((y_true == 0) & (y_pred == 1))[0]
    elif category == "false_negative": idx = np.where((y_true == 1) & (y_pred == 0))[0]
    else: raise ValueError(category)
    return paths[idx[0]] if len(idx) else None

ref = test_results[ANALYSIS_MODEL]
categories = ["correct_real", "correct_fake", "false_positive", "false_negative"]
gradcam_log_rows = []

for cat in categories:
    img_path = find_example(cat, ref["y_true"], ref["y_pred"], ref["paths"])
    if img_path is None:
        print(f"No example for '{cat}' - skipping."); continue
    fig, axes = plt.subplots(1, len(MODELS_TO_RUN) + 1, figsize=(4 * (len(MODELS_TO_RUN) + 1), 4.5))
    orig = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    axes[0].imshow(orig); axes[0].axis("off"); axes[0].set_title("Original")
    true_label = "FAKE" if "fake" in os.path.basename(os.path.dirname(img_path)).lower() else "REAL"
    for ax, name in zip(axes[1:], MODELS_TO_RUN):
        overlay, fake_prob = gradcam_for_image(trained_models[name], img_path)
        ax.imshow(overlay); ax.axis("off")
        pred_label = "FAKE" if fake_prob >= THRESHOLD else "REAL"
        ax.set_title(f"{name}\npred={pred_label}\nFAKE p={fake_prob:.3f}", fontsize=8)
        gradcam_log_rows.append({"category": cat, "image": os.path.basename(img_path), "model": name,
                                  "ground_truth": true_label, "prediction": pred_label,
                                  "fake_probability": fake_prob, "correct": pred_label == true_label})
    fig.suptitle(f"Figure 9: Grad-CAM comparison - {cat}"); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "gradcam", f"gradcam_{cat}.png"), dpi=300)
    plt.show()

gradcam_log_df = pd.DataFrame(gradcam_log_rows)
gradcam_log_df

## 22. Error Analysis (Figure 10)

In [ ]:
error_rows = []
for name in MODELS_TO_RUN:
    r = test_results[name]
    y_true, y_pred, y_prob, paths = r["y_true"], r["y_pred"], r["y_prob"], r["paths"]
    for i in range(len(y_true)):
        if y_true[i] != y_pred[i]:
            error_type = "False Positive (REAL->FAKE)" if y_true[i] == 0 else "False Negative (FAKE->REAL)"
            error_rows.append({"model": name, "image": paths[i],
                                "ground_truth": "REAL" if y_true[i] == 0 else "FAKE",
                                "prediction": "REAL" if y_pred[i] == 0 else "FAKE",
                                "probability_fake": y_prob[i], "error_type": error_type})
error_df = pd.DataFrame(error_rows)
error_df.to_csv(os.path.join(OUTPUT_DIR, "tables", "error_analysis.csv"), index=False)
print("Total errors:", len(error_df))
if len(error_df):
    display(error_df.groupby(["model", "error_type"]).size().unstack(fill_value=0))

model_errors = error_df[error_df["model"] == ANALYSIS_MODEL]
if len(model_errors):
    sample_errors = model_errors.head(4)
    fig, axes = plt.subplots(1, len(sample_errors), figsize=(4 * len(sample_errors), 4.5))
    if len(sample_errors) == 1: axes = [axes]
    for ax, (_, row) in zip(axes, sample_errors.iterrows()):
        overlay, fake_prob = gradcam_for_image(trained_models[ANALYSIS_MODEL], row["image"])
        ax.imshow(overlay); ax.axis("off")
        ax.set_title(f"{row['error_type']}\nGT:{row['ground_truth']} Pred:{row['prediction']}\np={fake_prob:.3f}", fontsize=8)
    fig.suptitle(f"Figure 10: Error analysis Grad-CAM - {ANALYSIS_MODEL}"); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "gradcam", f"{ANALYSIS_MODEL}_error_gradcam.png"), dpi=300)
    plt.show()
else:
    print("No errors found for", ANALYSIS_MODEL)

## 23. Confidence Analysis

In [ ]:
from sklearn.metrics import brier_score_loss

conf_summary = []
fig, axes = plt.subplots(1, len(MODELS_TO_RUN), figsize=(5 * len(MODELS_TO_RUN), 4))
if len(MODELS_TO_RUN) == 1: axes = [axes]
for ax, name in zip(axes, MODELS_TO_RUN):
    r = test_results[name]
    y_true, y_pred, y_prob = r["y_true"], r["y_pred"], r["y_prob"]
    conf = np.where(y_pred == 1, y_prob, 1 - y_prob)
    correct_mask = (y_true == y_pred)
    ax.hist(conf[correct_mask], bins=15, alpha=0.6, label="Correct", color="#4C72B0")
    ax.hist(conf[~correct_mask], bins=15, alpha=0.6, label="Incorrect", color="#C44E52")
    ax.set_title(name); ax.legend()
    brier = brier_score_loss(y_true, y_prob)
    n_bins = 10; bin_edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for b in range(n_bins):
        lo, hi = bin_edges[b], bin_edges[b + 1]
        in_bin = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        if in_bin.sum() == 0: continue
        ece += (in_bin.sum() / len(conf)) * abs(correct_mask[in_bin].mean() - conf[in_bin].mean())
    conf_summary.append({"Model": name, "Brier Score": brier, "ECE": ece})
plt.suptitle("Confidence distributions: correct vs incorrect"); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "confidence_distributions.png"), dpi=300)
plt.show()
conf_df = pd.DataFrame(conf_summary)
conf_df.to_csv(os.path.join(OUTPUT_DIR, "tables", "confidence_analysis.csv"), index=False)
conf_df

## 24. Robustness Experiment (Figure 11)

In [ ]:
import io, random as _random

RUN_ROBUSTNESS_TEST = True

def degrade_jpeg(pil_img, quality=25):
    buf = io.BytesIO(); pil_img.save(buf, format="JPEG", quality=quality); buf.seek(0)
    return Image.open(buf).convert("RGB")
def degrade_blur(pil_img, radius=3):
    return Image.fromarray(cv2.GaussianBlur(np.array(pil_img), (0, 0), sigmaX=radius))
def degrade_noise(pil_img, sigma=15):
    arr = np.array(pil_img).astype(np.float32)
    return Image.fromarray(np.clip(arr + np.random.normal(0, sigma, arr.shape), 0, 255).astype(np.uint8))
def degrade_resize(pil_img, factor=0.5):
    w, h = pil_img.size
    return pil_img.resize((max(1, int(w*factor)), max(1, int(h*factor)))).resize((w, h))

DEGRADATIONS = {"JPEG": degrade_jpeg, "Blur": degrade_blur, "Noise": degrade_noise, "Resize": degrade_resize}

@torch.no_grad()
def evaluate_degraded(model, samples, degrade_fn):
    model.eval(); probs, labels = [], []
    for path, label in samples:
        img = degrade_fn(Image.open(path).convert("RGB"))
        tensor = eval_transform(img).unsqueeze(0).to(DEVICE)
        probs.append(torch.sigmoid(model(tensor)).item()); labels.append(label)
    return np.array(probs), np.array(labels)

if RUN_ROBUSTNESS_TEST:
    ROBUSTNESS_SUBSET_SIZE = min(100, len(test_ds.samples))
    robustness_samples = _random.Random(RANDOM_SEED).sample(test_ds.samples, ROBUSTNESS_SUBSET_SIZE)
    robustness_rows = []
    for name in MODELS_TO_RUN:
        model = trained_models[name]
        base_probs, base_labels = evaluate_degraded(model, robustness_samples, lambda im: im)
        base_f1 = f1_score(base_labels, (base_probs >= THRESHOLD).astype(int), zero_division=0)
        row = {"Model": name, "Original F1": base_f1}
        for deg_name, deg_fn in DEGRADATIONS.items():
            d_probs, d_labels = evaluate_degraded(model, robustness_samples, deg_fn)
            d_f1 = f1_score(d_labels, (d_probs >= THRESHOLD).astype(int), zero_division=0)
            row[f"{deg_name} F1"] = d_f1
            row[f"{deg_name} % Drop"] = 100 * (base_f1 - d_f1) / base_f1 if base_f1 > 0 else float("nan")
        robustness_rows.append(row)
    robustness_df = pd.DataFrame(robustness_rows)
    robustness_df.to_csv(os.path.join(OUTPUT_DIR, "robustness", "robustness_results.csv"), index=False)
    display(robustness_df)

    fig, ax = plt.subplots(figsize=(9, 5))
    conditions = ["Original F1"] + [f"{d} F1" for d in DEGRADATIONS]
    x = np.arange(len(conditions)); width = 0.8 / len(MODELS_TO_RUN)
    for i, name in enumerate(MODELS_TO_RUN):
        row = robustness_df[robustness_df["Model"] == name].iloc[0]
        ax.bar(x + i * width, [row[c] for c in conditions], width, label=name)
    ax.set_xticks(x + width * (len(MODELS_TO_RUN) - 1) / 2)
    ax.set_xticklabels([c.replace(" F1", "") for c in conditions])
    ax.set_ylim(0, 1.05); ax.set_title("Figure 11: Robustness under degradation"); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plots", "fig11_robustness_comparison.png"), dpi=300)
    plt.show()
else:
    robustness_df = None

## 25. Statistical Comparison (McNemar's test)

In [ ]:
try:
    from statsmodels.stats.contingency_tables import mcnemar
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"], check=True)
    from statsmodels.stats.contingency_tables import mcnemar

mcnemar_df = pd.DataFrame()
if len(MODELS_TO_RUN) > 1:
    baseline_name = MODELS_TO_RUN[0]
    baseline_pred, baseline_true = test_results[baseline_name]["y_pred"], test_results[baseline_name]["y_true"]
    mcnemar_rows = []
    for name in MODELS_TO_RUN[1:]:
        other_pred, other_true = test_results[name]["y_pred"], test_results[name]["y_true"]
        assert np.array_equal(baseline_true, other_true), "Test label order mismatch."
        baseline_correct = (baseline_pred == baseline_true)
        other_correct = (other_pred == other_true)
        both_correct = int(np.sum(baseline_correct & other_correct))
        only_baseline = int(np.sum(baseline_correct & ~other_correct))
        only_other = int(np.sum(~baseline_correct & other_correct))
        both_wrong = int(np.sum(~baseline_correct & ~other_correct))
        table = [[both_correct, only_baseline], [only_other, both_wrong]]
        result = mcnemar(table, exact=(only_baseline + only_other) < 25, correction=True)
        mcnemar_rows.append({"Comparison": f"{baseline_name} vs {name}", "Statistic": result.statistic,
                              "p-value": result.pvalue, "Significant (alpha=0.05)": result.pvalue < 0.05})
    mcnemar_df = pd.DataFrame(mcnemar_rows)
    mcnemar_df.to_csv(os.path.join(OUTPUT_DIR, "tables", "mcnemar_results.csv"), index=False)
else:
    print("Only one model available - McNemar's test needs at least two.")
mcnemar_df

## 26. Final Comparison

In [ ]:
final_df = perf_df.merge(comp_df, on="Model").rename(columns={"Inference Time (ms/img)": "Inference Time (ms)"})
final_cols = ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Parameters", "Model Size (MB)", "Inference Time (ms)"]
final_df_display = final_df[final_cols]
final_df_display.to_csv(os.path.join(OUTPUT_DIR, "tables", "final_model_comparison.csv"), index=False)
final_df_display

## 27. Research Findings (auto-generated)

In [ ]:
def top_model(df, col, maximize=True):
    idx = df[col].idxmax() if maximize else df[col].idxmin()
    return df.loc[idx, "Model"], df.loc[idx, col]

best_acc_model, best_acc = top_model(perf_df, "Accuracy")
best_f1_model, best_f1 = top_model(perf_df, "F1")
best_auc_model, best_auc = top_model(perf_df, "ROC-AUC")
most_efficient_model, fastest_time = top_model(comp_df, "Inference Time (ms/img)", maximize=False)
smallest_model, smallest_size = top_model(comp_df, "Model Size (MB)", maximize=False)

findings_md = f"""### Automatically generated findings

- **Models evaluated this run:** {', '.join(MODELS_TO_RUN)}
- **Best accuracy:** {best_acc_model} ({best_acc:.4f})
- **Best F1:** {best_f1_model} ({best_f1:.4f})
- **Best ROC-AUC:** {best_auc_model} ({best_auc:.4f})
- **Fastest inference:** {most_efficient_model} ({fastest_time:.2f} ms/image)
- **Smallest model:** {smallest_model} ({smallest_size:.1f} MB)
"""
if robustness_df is not None:
    drop_cols = [c for c in robustness_df.columns if c.endswith("% Drop")]
    robustness_df["avg_pct_drop"] = robustness_df[drop_cols].mean(axis=1)
    most_robust_model = robustness_df.loc[robustness_df["avg_pct_drop"].idxmin(), "Model"]
    findings_md += f"- **Most robust to degradation:** {most_robust_model} (avg F1 drop {robustness_df['avg_pct_drop'].min():.1f}%)\n"
if len(mcnemar_df):
    sig = mcnemar_df[mcnemar_df["Significant (alpha=0.05)"]]
    findings_md += (f"- **Statistically significant vs baseline:** " + ", ".join(sig["Comparison"].tolist()) + "\n"
                     if len(sig) else "- No comparison reached statistical significance (alpha=0.05).\n")

from IPython.display import display, Markdown
display(Markdown(findings_md))
with open(os.path.join(OUTPUT_DIR, "tables", "research_findings.md"), "w") as f:
    f.write(findings_md)

## 28. Limitations

In [ ]:
limitations_md = f"""### Limitations

- **Single dataset.** All results come from one Kaggle dataset (manjilkarki/deepfake-and-real-images).
- **Limited manipulation diversity** relative to unseen deepfake generators.
- **Possible source/duplicate leakage** across splits was not verified with hashing/embedding checks.
- **Domain shift** to real deployment conditions is untested.
- **Grad-CAM limitations:** class-discriminative localization, not proof of causal reasoning.
- **Models evaluated this run:** only {', '.join(MODELS_TO_RUN)} — {"all four architectures" if len(MODELS_TO_RUN)==4 else "some checkpoints are still missing; re-run this notebook once the rest finish training"}.
- **Fixed decision threshold** ({THRESHOLD}) rather than per-model tuning.
"""
from IPython.display import display, Markdown
display(Markdown(limitations_md))
with open(os.path.join(OUTPUT_DIR, "tables", "limitations.md"), "w") as f:
    f.write(limitations_md)

## 29. Conclusion

In [ ]:
conclusion_md = f"""### Conclusion

1. **Best-performing model (this run):** {best_f1_model} (F1={best_f1:.4f}); {best_auc_model} had the best ROC-AUC ({best_auc:.4f}).
2. **Most efficient:** {most_efficient_model} ({fastest_time:.2f} ms/img); {smallest_model} was smallest on disk ({smallest_size:.1f} MB).
3. **Grad-CAM behavior:** see Section 20-21 figures for qualitative attention-pattern differences across models.
4. **Behavior under degradation:** {"see Section 24 - " + str(robustness_df.loc[robustness_df["avg_pct_drop"].idxmin(), "Model"]) + " degraded least on average." if robustness_df is not None else "robustness testing was not run."}
5. **Major failure modes:** see `error_df` (Section 22) for false positive/negative counts and Grad-CAM on those cases.
6. **Future work:** dataset leakage verification, cross-dataset generalization testing, per-model threshold tuning, quantitative Grad-CAM localization at scale.
"""
from IPython.display import display, Markdown
display(Markdown(conclusion_md))
with open(os.path.join(OUTPUT_DIR, "tables", "conclusion.md"), "w") as f:
    f.write(conclusion_md)
print("\\nAll tables/figures saved under:", OUTPUT_DIR)